In [45]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [46]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [67]:
df_ejercicios = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding = "utf-8")

In [68]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

## Incorporar clasificación de optimización

In [49]:
df_opt = pd.read_csv(root / "csv/ejercicios_optimizacion.csv", encoding = "utf-8")

In [70]:
df_enun = pd.read_csv(root / "csv/ejercicios_clasificados.csv", encoding = "utf-8")

In [51]:
df_opt.head(2)

,anio,convocatoria,ejercicio,enunciado_resumen,tema_optimizacion,categoria,max_min,cilindro
0,2007,Junio,Ejercicio 1 - Opción A,Dos números reales positivos cuya suma es 10 y...,Producto de los cuadrados de dos números con s...,Números,Máximo,No
1,2007,Reserva 1,Ejercicio 1 - Opción B,Dos chapas cuadradas de materiales a 2 y 3 €/c...,Coste total de dos chapas cuadradas con suma d...,Coste de materiales (figuras planas),Mínimo,No


In [71]:
df_enun.head(2)

,id,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen
0,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,NaN,Discuta la veracidad de las siguientes afirmac...,FISICA. 2025. JULIO. EJERCICIO a,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
1,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,2025,Julio,Ejercicio B1,campo eléctrico y magnético,NaN,Dos conductores rectilíneos muy largos se disp...,FISICA. 2025. JULIO. EJERCICIO b1,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...


In [53]:
df_opt = df_opt.iloc[:, :-2]

In [54]:
df_opt.loc[:, ["asignatura", "tema", "tipo_ejercicio"]] = ["Mates II", "funciones", "optimización"]

In [55]:
df_opt = df_opt.rename(columns = {
    "anio" : "año",
    "categoria" : "observaciones"
})

In [58]:
df_opt.columns

Index(['año', 'convocatoria', 'ejercicio', 'observaciones', 'asignatura',
       'tema', 'tipo_ejercicio'],
      dtype='str')

In [57]:
df_opt.drop(["enunciado_resumen", "tema_optimizacion"], axis = 1, inplace=True)

In [59]:
def create_id(row):
    row.ejercicio = row.ejercicio.replace(" - Opción", " opción")
    id = f"{row.asignatura}_{row.tema}_{row.año}_{row.convocatoria}_{row.ejercicio}".lower().replace(" ", "_")
    id = remove_accents(id)
    return id

In [60]:
df_opt["id"] = df_opt.apply(create_id, axis = 1)

In [61]:
df_opt.head(2)

,año,convocatoria,ejercicio,observaciones,asignatura,tema,tipo_ejercicio,id
0,2007,Junio,Ejercicio 1 - Opción A,Números,Mates II,funciones,optimización,mates_ii_funciones_2007_junio_ejercicio_1_opci...
1,2007,Reserva 1,Ejercicio 1 - Opción B,Coste de materiales (figuras planas),Mates II,funciones,optimización,mates_ii_funciones_2007_reserva_1_ejercicio_1_...


In [62]:
df_opt_to_merge = df_opt.loc[:, ["id", "tipo_ejercicio", "observaciones"]].set_index("id")

In [63]:
df_opt_to_merge.head()

,tipo_ejercicio,observaciones
id,,
mates_ii_funciones_2007_junio_ejercicio_1_opcion_a,optimización,Números
mates_ii_funciones_2007_reserva_1_ejercicio_1_opcion_b,optimización,Coste de materiales (figuras planas)
mates_ii_funciones_2007_reserva_2_ejercicio_1_opcion_a,optimización,Rectángulo inscrito entre gráficas/funciones
mates_ii_funciones_2007_reserva_4_ejercicio_1_opcion_b,optimización,Prisma/caja
mates_ii_funciones_2008_junio_ejercicio_1_opcion_b,optimización,Rectángulo (diagonal)


In [ ]:
df_enun = df_enun.set_index("id")

In [77]:
# añadir observaciones
df_enun["observaciones"] = pd.NA
df_enun.update(df_opt_to_merge)

In [78]:
df_enun.columns

Index(['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
       'tipo_ejercicio', 'enunciado', 'info_ejercicio', 'archivo', 'imagen',
       'observaciones'],
      dtype='str')

In [80]:
df_enun[(df_enun.asignatura == "Mates II") & (df_enun.tipo_ejercicio == "optimización")].sample(5)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,info_ejercicio,archivo,imagen,observaciones
id,,,,,,,,,,,
mates_ii_funciones_2010_reserva_1_ejercicio_1_opcion_a,Mates II,2010,Reserva 1,"Ejercicio 1, opción A",funciones,optimización,La hipotenusa de un triángulo rectángulo mide ...,MATEMÁTICAS II. 2010. RESERVA 1. EJERCICIO 1. ...,source_pdf_files/Mates II/funciones/2010 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...,Cono (sólido de revolución)
mates_ii_funciones_2017_junio_ejercicio_1_opcion_a,Mates II,2017,Junio,"Ejercicio 1, opción A",funciones,optimización,Se quiere hacer una puerta rectangular coronad...,MATEMÁTICAS II. 2017. JUNIO. EJERCICIO 1. OPCI...,source_pdf_files/Mates II/funciones/2017 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...,Ventana/puerta normanda
mates_ii_funciones_2022_reserva_1_ejercicio_2,Mates II,2022,Reserva 1,Ejercicio 2,funciones,optimización,Se quiere cercar un trozo de terreno como el d...,MATEMÁTICAS II. 2022. RESERVA 1. EJERCICIO 2,source_pdf_files/Mates II/funciones/2022 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...,Cercados y vallas
mates_ii_funciones_2011_reserva_1_ejercicio_1_opcion_a,Mates II,2011,Reserva 1,"Ejercicio 1, opción A",funciones,optimización,Un alambre de longitud 100 metros se divide en...,MATEMÁTICAS II. 2011. RESERVA 1. EJERCICIO 1. ...,source_pdf_files/Mates II/funciones/2011 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...,División de alambre/segmento (suma de áreas)
mates_ii_funciones_2007_reserva_1_ejercicio_1_opcion_b,Mates II,2007,Reserva 1,"Ejercicio 1,opción B",funciones,optimización,T enemos que fabricar dos chapas cuadradas con...,M ATEMÁTICAS II. 2007. RESERVA 1. EJERCICIO 1....,source_pdf_files/Mates II/funciones/2007 - Fun...,enunciados_ejercicios/Mates II/funciones/png/p...,Coste de materiales (figuras planas)


In [85]:
df_enun[df_enun.asignatura == "Mates II"].columns

Index(['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
       'tipo_ejercicio', 'enunciado', 'info_ejercicio', 'archivo', 'imagen',
       'observaciones'],
      dtype='str')

In [87]:
df_enun.reset_index(drop = False).to_csv(root / "csv/ejercicios_clasificados.csv", index = False, encoding="utf-8")